In [5]:
#importando bibliotecas
import re
from collections import Counter
import pandas as pd
import spacy
from spacy import displacy
from spacy.matcher import Matcher
from spacy.tokens import Span
from spacy.util import filter_spans
from spacy.language import Language
from pypdf import PdfReader

nlp = spacy.load("en_core_web_sm")

# texto curto
text = """Music is ubiquitous across human cultures—as a source of affective and pleasurable experience, moving us both physically and emotionally—and learning to play music shapes both brain structure and function. Music processing in the brain—namely, perception of melody, harmony, and rhythm—has traditionally been studied as an auditory phenomenon using passive listening paradigms. However, when listening to music, we actively generate predictions about what is likely to happen next. This enactive aspect has led to a more complete understanding of music processing involving brain structures implicated in action, emotion, and learning. Here, we review the cognitive neuroscience literature of music perception. We show that music perception, action, emotion, and learning all rest on the human brain’s fundamental capacity for prediction—as formulated by the predictive coding of music model. This review elucidates how this formulation of music perception and expertise in individuals can be extended to account for the dynamics and underlying brain mechanisms of collective music making. This in turn has important implications for human creativity as evinced by music improvisation. These recent advances shed new light on what makes music meaningful from a neuroscientific perspective."""

with open(r"C:\Users\marya\Downloads\music_in_the_brain.txt", "r", encoding="utf-8") as f:
    pdf_text = f.read()

pdf_text = pdf_text.replace("-\n", "").replace("\n", " ")   # tira quebras de linha
pdf_text = " ".join(pdf_text.split())                       # tira espaços repetidos
pdf_text = pdf_text.split("Boxes and figure captions")[0]   # fica só com o corpo do artigo

print(len(pdf_text))

62092


In [6]:
doc = nlp(text)

print(len(text), "caracteres")
print(len(doc), "tokens")
print(len(list(doc.sents)), "sentenças")

1289 caracteres
216 tokens
9 sentenças


In [7]:
#toker
print(text.split()[:8])
print([token.text for token in doc[:10]])

['Music', 'is', 'ubiquitous', 'across', 'human', 'cultures—as', 'a', 'source']
['Music', 'is', 'ubiquitous', 'across', 'human', 'cultures', '—', 'as', 'a', 'source']


In [8]:
# sentenças 
sentences = list(doc.sents)
for sentence in sentences:
    print(sentence.text[:100], "...")

Music is ubiquitous across human cultures—as a source of affective and pleasurable experience, movin ...
Music processing in the brain—namely, perception of melody, harmony, and rhythm—has traditionally be ...
However, when listening to music, we actively generate predictions about what is likely to happen ne ...
This enactive aspect has led to a more complete understanding of music processing involving brain st ...
Here, we review the cognitive neuroscience literature of music perception. ...
We show that music perception, action, emotion, and learning all rest on the human brain’s fundament ...
This review elucidates how this formulation of music perception and expertise in individuals can be  ...
This in turn has important implications for human creativity as evinced by music improvisation. ...
These recent advances shed new light on what makes music meaningful from a neuroscientific perspecti ...


In [9]:
# atributos de um token
token = [t for t in doc if t.text == "predictions"][0]

print(token.text)
print(token.head)
print(token.left_edge)
print(token.right_edge)
print(token.ent_type_)
print(token.ent_iob_)
print(token.lemma_)
print(token.morph)
print(token.pos_)
print(token.dep_)
print(token.lang_)

predictions
generate
predictions
next

O
prediction
Number=Plur
NOUN
dobj
en


In [10]:
# classe gramatical e dependência de cada token 
for token in sentences[2]:
    print(token.text, token.pos_, token.dep_)

However ADV advmod
, PUNCT punct
when SCONJ advmod
listening VERB advcl
to ADP prep
music NOUN pobj
, PUNCT punct
we PRON nsubj
actively ADV advmod
generate VERB ROOT
predictions NOUN dobj
about ADP prep
what PRON nsubj
is AUX pcomp
likely ADJ acomp
to PART aux
happen VERB xcomp
next ADV advmod
. PUNCT punct


In [11]:
displacy.render(sentences[2], style="dep", jupyter=True)

In [12]:
# abstract
for ent in doc.ents:
    print(ent.text, ent.label_)
print("total:", len(doc.ents))

total: 0


In [13]:
# artigo inteiro
doc_pdf = nlp(pdf_text)

for label, n in Counter(ent.label_ for ent in doc_pdf.ents).most_common():
    print(label, n, spacy.explain(label))

CARDINAL 98 Numerals that do not fall under another type
ORG 91 Companies, agencies, institutions, etc.
PERSON 31 People, including fictional
DATE 18 Absolute or relative dates or periods
NORP 13 Nationalities or religious or political groups
GPE 9 Countries, cities, states
PRODUCT 6 Objects, vehicles, foods, etc. (not services)
ORDINAL 5 "first", "second", etc.
LAW 3 Named documents made into laws.
TIME 2 Times smaller than a day
QUANTITY 1 Measurements, as of weight or distance
LANGUAGE 1 Any named language
PERCENT 1 Percentage, including "%"
LOC 1 Non-GPE locations, mountain ranges, bodies of water


In [14]:
print(Counter(ent.text for ent in doc_pdf.ents if ent.label_ == "ORG").most_common(8))
print(Counter(ent.text for ent in doc_pdf.ents if ent.label_ == "PERSON" and len(ent.text.split()) >= 2).most_common(8))

[('PCM', 22), ('MMN', 11), ('NAcc', 9), ('EEG', 5), ('IFG', 4), ('MEG', 3), ('ears4', 2), ('Cheung', 2)]
[('Box 2', 2), ('Peter Vuust', 1), ('Ole A. Heggli', 1), ('Karl J. Friston &', 1), ('Pat Metheny', 1), ('Phil Collins', 1), ('Arnold Schoenberg', 1), ('Rhythm Perception', 1)]


In [15]:
exemplo = nlp("Pat Metheny said that listening is the key. Phil Collins played a drum fill in London.")
displacy.render(exemplo, style="ent", jupyter=True)

In [16]:
nlp_md = spacy.load("en_core_web_md")

doc1 = nlp_md("I enjoy listening to jazz music.")
doc2 = nlp_md("She loves classical songs and concerts.")
doc3 = nlp_md("The car needs new tires and an oil change.")

print(doc1.similarity(doc2))   # mesmo assunto
print(doc1.similarity(doc3))   # assunto diferente

0.8084021806716919
0.5661834478378296


In [18]:
# palavras
print(nlp_md("brain")[0].similarity(nlp_md("cortex")[0]))
print(nlp_md("brain")[0].similarity(nlp_md("banana")[0]))
print(nlp_md("dog")[0].similarity(nlp_md("cat")[0]))

0.5535789132118225
0.07295036315917969
1.0000001192092896


In [21]:
nlp_ruler = spacy.load("en_core_web_sm")
ruler = nlp_ruler.add_pipe("entity_ruler", before="ner")

patterns = [
    {"label": "MODEL",  "pattern": "predictive coding of music"},
    {"label": "MODEL",  "pattern": "PCM"},
    {"label": "METHOD", "pattern": "EEG"},
    {"label": "METHOD", "pattern": "MEG"},
    {"label": "METHOD", "pattern": "fMRI"},
    {"label": "ERP",    "pattern": "MMN"},
    {"label": "ERP",    "pattern": "ERAN"},
    {"label": "CONCEPT", "pattern": [{"LOWER": "prediction"}, {"LOWER": {"IN": ["error", "errors"]}}]},
]
ruler.add_patterns(patterns)

doc_ruler = nlp_ruler(text)
for ent in doc_ruler.ents:
    print(ent.text, ent.label_)

predictive coding of music MODEL


In [23]:
# no artigo inteiro
doc_ruler = nlp_ruler(pdf_text)
print(Counter((ent.text.lower(), ent.label_) for ent in doc_ruler.ents if ent.label_ in {"MODEL", "METHOD", "ERP", "CONCEPT"}).most_common(10))

[(('pcm', 'MODEL'), 24), (('prediction error', 'CONCEPT'), 23), (('prediction errors', 'CONCEPT'), 13), (('mmn', 'ERP'), 12), (('eeg', 'METHOD'), 6), (('eran', 'ERP'), 6), (('fmri', 'METHOD'), 6), (('predictive coding of music', 'MODEL'), 4), (('meg', 'METHOD'), 3)]


In [24]:
sentence = "Pat Metheny said that listening is the key."

nlp_a = spacy.load("en_core_web_sm")
nlp_a.add_pipe("entity_ruler", before="ner").add_patterns([{"label": "MUSICIAN", "pattern": "Pat Metheny"}])

nlp_b = spacy.load("en_core_web_sm")
nlp_b.add_pipe("entity_ruler", after="ner").add_patterns([{"label": "MUSICIAN", "pattern": "Pat Metheny"}])

print([(e.text, e.label_) for e in nlp_a(sentence).ents])
print([(e.text, e.label_) for e in nlp_b(sentence).ents])

[('Pat Metheny', 'MUSICIAN')]
[('Pat Metheny', 'PERSON')]


In [25]:
matcher = Matcher(nlp.vocab)

matcher.add("ADJ_NOUN", [[{"POS": "ADJ", "OP": "+"}, {"POS": "NOUN", "OP": "+"}]], greedy="LONGEST")
matcher.add("X_PERCEPTION", [[{"POS": "NOUN"}, {"LOWER": "perception"}]])
matcher.add("PREDICTION", [[{"LEMMA": {"IN": ["predict", "prediction"]}}]])

matches = matcher(doc)
matches.sort(key=lambda x: x[1])

for match_id, start, end in matches:
    print(nlp.vocab.strings[match_id], "->", doc[start:end].text)

ADJ_NOUN -> human cultures
ADJ_NOUN -> pleasurable experience
ADJ_NOUN -> auditory phenomenon
ADJ_NOUN -> passive listening paradigms
PREDICTION -> predictions
ADJ_NOUN -> enactive aspect
ADJ_NOUN -> complete understanding
ADJ_NOUN -> cognitive neuroscience literature
X_PERCEPTION -> music perception
X_PERCEPTION -> music perception
ADJ_NOUN -> human brain
ADJ_NOUN -> fundamental capacity
PREDICTION -> prediction
ADJ_NOUN -> predictive coding
X_PERCEPTION -> music perception
ADJ_NOUN -> collective music
ADJ_NOUN -> important implications
ADJ_NOUN -> human creativity
ADJ_NOUN -> recent advances
ADJ_NOUN -> new light
ADJ_NOUN -> neuroscientific perspective


In [26]:
matcher2 = Matcher(nlp.vocab)
matcher2.add("PC", [[
    {"LOWER": "predictive"},
    {"LOWER": "coding"},
    {"LOWER": "of", "OP": "?"},
    {"LOWER": "music", "OP": "?"},
]], greedy="LONGEST")

for match_id, start, end in matcher2(doc):
    print(doc[start:end].text)

predictive coding of music


In [27]:
#pares adjetivo e substantivo mais comuns
matcher3 = Matcher(nlp.vocab)
matcher3.add("ADJ_NOUN", [[{"POS": "ADJ"}, {"POS": "NOUN"}]])

pairs = Counter(doc_pdf[start:end].text.lower() for match_id, start, end in matcher3(doc_pdf))
print(pairs.most_common(10))

[('predictive coding', 17), ('musical training', 10), ('predictive processing', 8), ('musical improvisation', 8), ('predictive model', 7), ('predictive brain', 6), ('active inference', 6), ('predictive mechanisms', 5), ('western music', 5), ('musical pleasure', 5)]


In [28]:
sample = "The MMN peaks 110-250 ms after change onset, and the ERAN peaks at 150-200 ms."
pattern = r"\d{2,3}-\d{2,3} ms"

print(re.findall(pattern, sample))

for hit in re.finditer(pattern, sample):
    print(hit.start(), hit.end(), sample[hit.start():hit.end()])

['110-250 ms', '150-200 ms']
14 24 110-250 ms
67 77 150-200 ms


In [29]:
#entidade
doc_regex = doc_pdf.copy()
pattern = r"\b\d{2,3}(?:-\d{2,3})? ms\b"

original_ents = list(doc_regex.ents)
new_ents = []

for hit in re.finditer(pattern, doc_regex.text):
    start, end = hit.span()
    span = doc_regex.char_span(start, end)
    if span is not None:
        new_ents.append(Span(doc_regex, span.start, span.end, label="LATENCY"))

doc_regex.ents = filter_spans(new_ents + original_ents)

print([ent.text for ent in doc_regex.ents if ent.label_ == "LATENCY"])

['150-200 ms']


In [30]:
# erro de sobreposição e filter_spans
d = nlp("The mismatch negativity peaks early.")
span1 = Span(d, 1, 3, label="ERP")
span2 = Span(d, 2, 4, label="OTHER")

try:
    d.ents = [span1, span2]
except ValueError as e:
    print("erro:", str(e)[:70])

d.ents = filter_spans([span1, span2])
print([(e.text, e.label_) for e in d.ents])

erro: [E1010] Unable to set entity information for token 2 which is included
[('mismatch negativity', 'ERP')]


In [31]:
df = pd.DataFrame({
    "Name":   ["amygdala", "hippocampus", "nucleus accumbens", "orbitofrontal cortex",
               "inferior frontal gyrus", "planum temporale", "supplementary motor area",
               "basal ganglia", "cerebellum", "premotor cortex", "insula"],
    "Symbol": [None, None, "NAcc", "OFC", "IFG", "PT", "SMA", None, None, "PMC", None],
})
df

,Name,Symbol
0,amygdala,NaN
1,hippocampus,NaN
2,nucleus accumbens,NAcc
3,orbitofrontal cortex,OFC
4,inferior frontal gyrus,IFG
5,planum temporale,PT
6,supplementary motor area,SMA
7,basal ganglia,NaN
8,cerebellum,NaN
9,premotor cortex,PMC


In [32]:
nlp_regions = spacy.blank("en")
ruler = nlp_regions.add_pipe("entity_ruler")

patterns = []
for name, symbol in zip(df.Name, df.Symbol):
    patterns.append({"label": "BRAIN_REGION", "pattern": name})
    patterns.append({"label": "BRAIN_REGION", "pattern": name.capitalize()})
    if symbol:
        patterns.append({"label": "BRAIN_REGION", "pattern": symbol})

ruler.add_patterns(patterns)
print(len(patterns), "padrões")

doc_regions = nlp_regions(pdf_text)
print(Counter(ent.text.lower() for ent in doc_regions.ents).most_common(10))

33 padrões
[('nacc', 9), ('ifg', 5), ('pt', 3), ('amygdala', 3), ('hippocampus', 3), ('pmc', 2), ('sma', 2), ('planum temporale', 1), ('inferior frontal gyrus', 1), ('nucleus accumbens', 1)]


In [33]:
# conferir o contexto das siglas curtas 
for ent in doc_regions.ents:
    if ent.text in ("PT", "SMA"):
        print(ent.text, "...", doc_regions.text[max(0, ent.start_char - 40): ent.end_char + 40])

PT ... excitatory connectivity from left A1 to PT, has been interpreted as the passing of
PT ... gests that pitch height is processed in PT posterior to A1, and chroma change in t
PT ... , projecting from primary areas via the PT over the parietal cortex to the dorsola
SMA ... he premotor (PMC), supplementary motor (SMA), basal ganglia, and cerebellum106,107 
SMA ... efrontal brain regions, such as the pre-SMA, medial prefrontal cortex (mPFC), IFG, 


In [34]:
#LISTEN e SILENT ficam com os mesmos números
print([ord(c) for c in "LISTEN"])
print([ord(c) for c in "SILENT"])

[76, 73, 83, 84, 69, 78]
[83, 73, 76, 69, 78, 84]


In [35]:
try:
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
except Exception:
    # Keras novo não tem mais o Tokenizer
    from keras_preprocessing.text import Tokenizer
    import numpy as np

    def pad_sequences(sequences, maxlen):
        padded = np.zeros((len(sequences), maxlen), dtype=int)
        for i, seq in enumerate(sequences):
            seq = seq[-maxlen:]
            padded[i, maxlen - len(seq):] = seq
        return padded

In [36]:
sentences = [
    'i love my dog',
    'I, love my cat',
    'You love my dog!'
]

tokenizer = Tokenizer(num_words=100)
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index
print(word_index)

{'love': 1, 'my': 2, 'i': 3, 'dog': 4, 'cat': 5, 'you': 6}


In [37]:
# agora com as sentenças do abstract
sentences = [s.text for s in doc.sents]

tokenizer = Tokenizer(num_words=200, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index

print(len(word_index), "palavras")
print(list(word_index.items())[:12])

120 palavras
[('<OOV>', 1), ('music', 2), ('and', 3), ('of', 4), ('to', 5), ('the', 6), ('in', 7), ('perception', 8), ('this', 9), ('human', 10), ('a', 11), ('learning', 12)]


In [38]:
sequences = tokenizer.texts_to_sequences(sentences)
print(sequences[0])

[2, 16, 28, 29, 10, 30, 11, 31, 4, 32, 3, 33, 34, 35, 36, 17, 37, 3, 38, 12, 5, 39, 2, 40, 17, 13, 41, 3, 42]


In [39]:
# palavras desconhecidas viram 
test_data = ["Music shapes the brain", "Jazz improvisation is creative"]
print(tokenizer.texts_to_sequences(test_data))

[[2, 40, 6, 13], [1, 109, 16, 1]]


In [40]:
padded = pad_sequences(sequences, maxlen=40)
print(padded.shape)
print(padded[0])

(9, 40)
[ 0  0  0  0  0  0  0  0  0  0  0  2 16 28 29 10 30 11 31  4 32  3 33 34
 35 36 17 37  3 38 12  5 39  2 40 17 13 41  3 42]
